# Bromobutane RESP Example
## Without Extra Points, and
## With an Extra Point that Models a Sigma Hole

---

Specs:
- **2 conformation**
- 1 spatial orientation
- Two-stage RESP is performed
    - Stage 1: stage_1.ini
    - Stage 2: stage_2.ini

---

In [ ]:
from pathlib import Path

import numpy as np

from resp_ep import driver
from resp_ep import extras

In [ ]:
def charges2freeze_format(atoms: list, charges_array: np.array):
    """ Formats and prints partial charges for specific atoms as constraints.

        This function extracts atomic charges from a multidimensional array 
        and formats them into a specific string structure required for freezing 
        charges during geometry or charge optimizations.

        Args:
            atoms (list): A list of atom numbers (1-indexed) to format.
    
        Globals Used:
            freeze_charges (iterable): 1-indexed atom numbers to look up.
            charges_stage_1_x (list/tuple of np.ndarray): Structure holding the
                charge arrays, where index [1] contains the target stage charges.

        Returns:
            None: The function prints the formatted string directly to stdout.

        Example:
            >>> charges2freeze_format([5, 6, 7])
            constraint_charge :  5 =  0.09735144,
                                 6 = -0.01063565,
                                 7 = -0.00799172
    """
    lines = []
    for atom in atoms:
        index = atom - 1
        charge = charges_array[1][index]
        lines.append(f"{atom:>3} = {charge: .8f}")

    output = "constraint_charge :" + ",\n                   ".join(lines)

    print(output)

## Add Sigma Hole

In [ ]:
extras.add_sigma_hole(molecule='1_bromobutane_conf1', atom_pairs=[[11, 14]], distance=1.3)
extras.add_sigma_hole(molecule='1_bromobutane_conf2', atom_pairs=[[11, 14]], distance=1.3)

## Stage 1

In [ ]:
stage_1 = f'''\
[molecules]
input_files: 1_bromobutane_conf1.xyz,
             1_bromobutane_conf2.xyz
    
[vdw.surface.options]
vdw_scale_factors : 1.4, 1.6, 1.8, 2.0
vdw_point_density : 1.0
esp               : None
grid              : None

vdw_radii_set     : legacy
vdw_radii         : BR = 1.85

[hyperbolic.restraint.options]
weight            : 1.0, 1.0
restraint         : True
resp_a            : 0.0005
resp_b            : 0.1
ihfree            : True
toler             : 1e-5
max_it            : 50 

[constraints]
constraint_charge : None
equivalent_groups : None

[qm.options]
method_esp : hf
basis_esp  : assign H 6-31G*,
             assign C 6-31G*,
             assign O 6-31G*,
             assign Br 6-31G*

formal_charge : 0
multiplicity  : 1

'''

with open('stage_1.ini', 'w') as file:
    file.write(stage_1)

#### Extra Points

In [ ]:
stage_1_x = f'''\
[molecules]
input_files: 1_bromobutane_conf1_x.xyz,
             1_bromobutane_conf2_x.xyz
    
[vdw.surface.options]
vdw_scale_factors : 1.4, 1.6, 1.8, 2.0
vdw_point_density : 1.0
esp               : 1_bromobutane_conf1_esp.dat,
                    1_bromobutane_conf2_esp.dat,

grid              : 1_bromobutane_conf1_grid.dat,
                    1_bromobutane_conf2_grid.dat

vdw_radii_set     : legacy
vdw_radii         : BR = 1.85,
                    X = 0.0

[hyperbolic.restraint.options]
weight            : 1.0, 1.0
restraint         : True
resp_a            : 0.0005
resp_b            : 0.1
ihfree            : True
toler             : 1e-5
max_it            : 50 

[constraints]
constraint_charge : None
equivalent_groups : None

[qm.options]
method_esp : None
basis_esp  : None

formal_charge : 0
multiplicity  : 1
'''

with open('stage_1_x.ini', 'w') as file:
    file.write(stage_1_x)

In [ ]:
charges_stage_1 = driver.resp(input_ini='stage_1.ini')
print()
print(f'ESP:\n{charges_stage_1[0]}\n')
print(f'RESP Stage 1:\n{charges_stage_1[1]}')

In [ ]:
charges_stage_1_x = driver.resp(input_ini='stage_1_x.ini')
print()
print(f'ESP:\n{charges_stage_1_x[0]}\n')
print(f'RESP Stage 1:\n{charges_stage_1_x[1]}')

### Extract charges from stage 1 that you want to freeze

In [ ]:
charges2freeze_format(atoms = [14], charges_array=charges_stage_1) #Atom/Extra Point #s

In [ ]:
charges2freeze_format(atoms = [14, 15], charges_array=charges_stage_1_x) #Atom/Extra Point #s

Copy and paste the above lines in the INI files below.

## Stage 2

In [ ]:
stage_2 = f'''\
[molecules]
input_files: 1_bromobutane_conf1.xyz,
             1_bromobutane_conf2.xyz

[vdw.surface.options]
vdw_scale_factors : 1.4, 1.6, 1.8, 2.0
vdw_point_density : 1.0
esp               : 1_bromobutane_conf1_esp.dat,
                    1_bromobutane_conf2_esp.dat,

grid              : 1_bromobutane_conf1_grid.dat,
                    1_bromobutane_conf2_grid.dat

vdw_radii_set     : legacy
vdw_radii         : BR = 1.85

[hyperbolic.restraint.options]
weight            : 1.0, 1.0
restraint         : True
resp_a            : 0.001
resp_b            : 0.1
ihfree            : True
toler             : 1e-5
max_it            : 50 

[constraints]
constraint_charge : 14 = -0.20806479

equivalent_groups : group_1 = 2 3 4,
                    group_2 = 6 7,
                    group_3 = 9 10,
                    group_4 = 12 13

[qm.options]
method_esp    : None
basis_esp     : None

formal_charge : 0
multiplicity  : 1
'''

with open('stage_2.ini', 'w') as file:
    file.write(stage_2)

In [ ]:
stage_2_x = f'''\
[molecules]
input_files: 1_bromobutane_conf1_x.xyz,
             1_bromobutane_conf2_x.xyz

[vdw.surface.options]
vdw_scale_factors : 1.4, 1.6, 1.8, 2.0
vdw_point_density : 1.0
esp               : 1_bromobutane_conf1_esp.dat,
                    1_bromobutane_conf2_esp.dat,

grid              : 1_bromobutane_conf1_grid.dat,
                    1_bromobutane_conf2_grid.dat

vdw_radii_set     : legacy
vdw_radii         : BR = 1.85,
                    X = 0.0

[hyperbolic.restraint.options]
weight            : 1.0, 1.0
restraint         : True
resp_a            : 0.0005
resp_b            : 0.1
ihfree            : True
toler             : 1e-5
max_it            : 50 

[constraints]
constraint_charge : 14 = -0.41555262,
                    15 =  0.10296639

equivalent_groups : group_1 = 2 3 4,
                    group_2 = 6 7,
                    group_3 = 9 10,
                    group_4 = 12 13

[qm.options]
method_esp : None
basis_esp  : None

formal_charge : 0
multiplicity  : 1
'''

with open('stage_2_x.ini', 'w') as file:
    file.write(stage_2_x)

In [ ]:
charges_stage_2 = driver.resp(input_ini='stage_2.ini')
print()
print(f'ESP:\n{charges_stage_2[0]}\n')
print(f'RESP Stage 2:\n{charges_stage_2[1]}')

In [ ]:
charges_stage_2_x = driver.resp(input_ini='stage_2_x.ini')
print()
print(f'ESP:\n{charges_stage_2_x[0]}\n')
print(f'RESP Stage 2:\n{charges_stage_2_x[1]}')